# 🎓 UPPA Smart Assistant · Notebook 02 V3
## **Optimiser le chunking et la recherche multi-domaines — expérience comparative**

**Pour les deux masters M2 :** Industry 4.0 · Systèmes informatiques pour le génie de la logistique industrielle et des services.

**Trois domaines :** Formations · Admissions · Recherche. Ce notebook est un **atelier supplémentaire**, il ne remplace **pas** le Notebook 02 V2.

> Question scientifique : *une meilleure préparation documentaire et une recherche spécialisée améliorent-elles la pertinence des informations retrouvées ?*

**Entrées réelles recommandées (deux archives) :**
- `UPPA_Notebook01_exports_uppa.zip` : corpus source non découpé ;
- `UPPA_Notebook02_index_uppa.zip` : chunks **exactement produits par V2**, servant de référence expérimentale.

**Sorties :** deux collections distinctes (V2 reproduite à partir des chunks exportés, V3 optimisée), rapport de qualité, grille d'annotation, comparaison document-level et archive V3 compatible avec le Notebook 03.

⚠️ Le mode **fictif** sert uniquement à tester les méthodes. L'absence d'annotations humaines interdit de conclure à une amélioration statistiquement établie. Les résultats d'admission restent à vérifier sur les pages institutionnelles à jour.

### Objectifs d'apprentissage

1. Charger et contrôler **deux livrables préexistants** sans réexplorer le Web.
2. Diagnostiquer le bruit documentaire et la troncature en tokens.
3. Comparer un index V2 de référence et un nouvel index V3, avec **le même modèle d'embeddings**.
4. Appliquer filtres, quotas par document et décomposition raisonnée de questions transversales.
5. Évaluer au niveau des **documents sources** (Hit@k, Recall@k, MRR@k), après annotation humaine.
6. Exporter les résultats pour le Notebook 03.

**Méthode expérimentale :** A = V2 + recherche globale ; B = V3 + recherche globale ; C = V3 + diversification ; D = V3 + sous-questions spécialisées (D0 = la même sans les bonus manuels d'admission).

In [ ]:
from IPython.display import display, HTML

display(HTML("""<div style="border-left:5px solid #ae6346;background:#f5f2eb;padding:16px;border-radius:12px;font-family:Arial,sans-serif;color:#203a32"><b>Atelier d'expérimentation</b> · L'objectif n'est pas de maximiser artificiellement les scores, mais de comprendre <i>pourquoi</i> les résultats changent. Aucune page UPPA n'est téléchargée par ce notebook.</div>"""))

---
## Étape 0 — Installer l'environnement Colab

Qdrant fonctionne **localement** (sans Docker, sans clé API). Les modèles sont téléchargés lors de la première exécution ; prévoir une connexion Internet dans Colab et, si possible, un GPU. Le navigateur de fichiers Colab permettra de téléverser vos archives V1/V2.

In [ ]:
%pip -q install "sentence-transformers>=3,<6" "qdrant-client>=1.12,<2" "langchain-text-splitters>=0.3,<2" "pandas>=2" "matplotlib>=3.7"

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from hashlib import sha256
from urllib.parse import urlparse
from IPython.display import display
import json, re, zipfile, uuid, shutil, io, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

BASE = Path('/content/UPPA_RAG_NB02_V3' if Path('/content').exists() else './UPPA_RAG_NB02_V3').resolve()
EXPORT = BASE/'exports'
DB = BASE/'qdrant_local'
for d in (BASE,EXPORT,DB): d.mkdir(parents=True,exist_ok=True)
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'  # identique à V2
DOMAINS = ('formation','admission','recherche')
print('Dossier indépendant de V2 :',BASE)

---
## Étape 1 — Charger le corpus brut et l'index de référence V2

**Pourquoi deux archives ?** Le ZIP V1 contient le texte complet pour refaire le chunking ; le ZIP V2 contient les véritables chunks de la baseline. Si vous ne fournissez que V1, les résultats ne seront pas présentés comme une comparaison avec l'index V2 original.

- Dans Colab : mettez `UPLOAD_WITH_COLAB=True` et sélectionnez les **deux archives**.
- Si les fichiers se trouvent déjà dans `/content`, le notebook tente de les retrouver.
- Sans fichiers, il utilise six fiches **fictives** et construit une baseline de démonstration (non comparable à votre exécution réelle de V2).

In [ ]:
NB01_PATH = ''  # ex. '/content/UPPA_Notebook01_exports_uppa.zip'
NB02_V2_PATH = '' # ex. '/content/UPPA_Notebook02_index_uppa.zip'
UPLOAD_WITH_COLAB = False

if UPLOAD_WITH_COLAB:
    from google.colab import files
    uploaded=files.upload()   # sélectionner les deux fichiers en une fois
    print('Fichiers reçus :',list(uploaded))

def find_input(pattern):
    roots=[Path('/content'),Path.cwd()]
    hits=[]
    for root in roots:
        if root.is_dir(): hits.extend(root.glob(pattern))
    # préférer le corpus réel au fichier fictif
    hits.sort(key=lambda p:('synthetic_demo' in p.name,-p.stat().st_mtime))
    return str(hits[0]) if hits else ''

if not NB01_PATH: NB01_PATH=find_input('UPPA_Notebook01_exports_*.zip') or find_input('corpus_uppa.jsonl')
if not NB02_V2_PATH: NB02_V2_PATH=find_input('UPPA_Notebook02_index_*.zip')
print('NB01 :',NB01_PATH or '(absent : démonstration fictive)')
print('NB02 V2 :',NB02_V2_PATH or '(absent : aucune comparaison réelle V2)')

In [ ]:
DEMO_DOCUMENTS = json.loads(r'''[
  {
    "document_id": "demo_ffe9847450c745d3",
    "domain": "formation",
    "document_type": "fiction",
    "title": "Master informatique — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Master informatique — exemple fictif"
      },
      {
        "level": 2,
        "text": "Présentation"
      },
      {
        "level": 2,
        "text": "Parcours"
      },
      {
        "level": 2,
        "text": "Compétences"
      }
    ],
    "links": [],
    "table_count": 0,
    "content_markdown": "# Master informatique — exemple fictif\n\n## Présentation\n\nFormation expérimentale qui illustre les systèmes d'information, l'IA et l'informatique industrielle.\n\n## Parcours\n\n* Industry 4.0 — cas fictif\n* Logistique et services — cas fictif\n\n## Compétences\n\nConcevoir des applications et analyser des données.",
    "content_text": "Master informatique — exemple fictif Présentation Formation expérimentale qui illustre les systèmes d'information, l'IA et l'informatique industrielle. Parcours Industry 4.0 — cas fictif Logistique et services — cas fictif Compétences Concevoir des applications et analyser des données.",
    "sha256_text": "23fe2e187b2e3925a1ebf92371bcc7c2d08bf38b09896ab9e28d8c6e7769df3c",
    "character_count": 286,
    "word_count": 37
  },
  {
    "document_id": "demo_494779650bc67814",
    "domain": "formation",
    "document_type": "fiction",
    "title": "Parcours Industry 4.0 — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Parcours Industry 4.0 — exemple fictif"
      },
      {
        "level": 2,
        "text": "Objectifs"
      },
      {
        "level": 2,
        "text": "Tableau des unités"
      }
    ],
    "links": [],
    "table_count": 1,
    "content_markdown": "# Parcours Industry 4.0 — exemple fictif\n\n## Objectifs\n\nExplorer la maintenance prédictive, les capteurs et le pilotage de production.\n\n## Tableau des unités\n\n| UE | Thème |\n| --- | --- |\n| UE1 | Capteurs |\n| UE2 | Données industrielles |",
    "content_text": "Parcours Industry 4.0 — exemple fictif Objectifs Explorer la maintenance prédictive, les capteurs et le pilotage de production. Tableau des unités UE Thème UE1 Capteurs UE2 Données industrielles",
    "sha256_text": "374260d1e7cf7469f0b9362b90e1098f570027f0e31ef7019c2cd26a708b15e4",
    "character_count": 194,
    "word_count": 28
  },
  {
    "document_id": "demo_c593913670f65f7d",
    "domain": "admission",
    "document_type": "fiction",
    "title": "Admission en master — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Admission en master — exemple fictif"
      },
      {
        "level": 2,
        "text": "Documents"
      },
      {
        "level": 2,
        "text": "Procédure"
      }
    ],
    "links": [],
    "table_count": 0,
    "content_markdown": "# Admission en master — exemple fictif\n\n## Documents\n\n* Curriculum vitae\n* Relevés de notes\n\n## Procédure\n\nConsulter les modalités officielles, différentes selon le profil et l'année.\n\nLes dates citées dans ce document sont fictives.",
    "content_text": "Admission en master — exemple fictif Documents Curriculum vitae Relevés de notes Procédure Consulter les modalités officielles, différentes selon le profil et l'année. Les dates citées dans ce document sont fictives.",
    "sha256_text": "dc105ec8fbb47a89d7e54543798866444e2a09f885ad9f2050c74d868c263c96",
    "character_count": 216,
    "word_count": 31
  },
  {
    "document_id": "demo_d1e2dbcf93b1520d",
    "domain": "admission",
    "document_type": "fiction",
    "title": "Calendrier d'admission — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Calendrier d'admission — exemple fictif"
      },
      {
        "level": 2,
        "text": "Dates illustratives"
      }
    ],
    "links": [],
    "table_count": 1,
    "content_markdown": "# Calendrier d'admission — exemple fictif\n\n## Dates illustratives\n\n| Phase | Date fictive |\n| --- | --- |\n| Candidature | 15 janvier 2030 |\n| Réponse | 20 mars 2030 |\n\nCes dates ne concernent aucune campagne officielle.",
    "content_text": "Calendrier d'admission — exemple fictif Dates illustratives Phase Date fictive Candidature 15 janvier 2030 Réponse 20 mars 2030 Ces dates ne concernent aucune campagne officielle.",
    "sha256_text": "6e16b05ee700b7b029fefc5fa48ee8283e18dc69b07f6f5d7128b2f265ab1f09",
    "character_count": 179,
    "word_count": 25
  },
  {
    "document_id": "demo_432ebb2fd9f03e32",
    "domain": "recherche",
    "document_type": "fiction",
    "title": "Laboratoire Systèmes numériques — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Laboratoire Systèmes numériques — exemple fictif"
      },
      {
        "level": 2,
        "text": "Axes de recherche"
      },
      {
        "level": 2,
        "text": "Équipes"
      }
    ],
    "links": [],
    "table_count": 0,
    "content_markdown": "# Laboratoire Systèmes numériques — exemple fictif\n\n## Axes de recherche\n\nRéseaux, intelligence artificielle, simulation et systèmes cyber-physiques.\n\n## Équipes\n\nEquipe Méthodes et Données.",
    "content_text": "Laboratoire Systèmes numériques — exemple fictif Axes de recherche Réseaux, intelligence artificielle, simulation et systèmes cyber-physiques. Équipes Equipe Méthodes et Données.",
    "sha256_text": "418165d42fc1434b2e1deed1aaa7bbb3422d4587787cd75a442734c366c8857a",
    "character_count": 178,
    "word_count": 21
  },
  {
    "document_id": "demo_f52ce7bf39c1439e",
    "domain": "recherche",
    "document_type": "fiction",
    "title": "Projet Logistique intelligente — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Projet Logistique intelligente — exemple fictif"
      },
      {
        "level": 2,
        "text": "Thématiques"
      },
      {
        "level": 2,
        "text": "Partenariats"
      }
    ],
    "links": [],
    "table_count": 0,
    "content_markdown": "# Projet Logistique intelligente — exemple fictif\n\n## Thématiques\n\n* Traçabilité\n* Optimisation d'entrepôt\n* Recommandation de services\n\n## Partenariats\n\nExemple pédagogique sans partenaires réels.",
    "content_text": "Projet Logistique intelligente — exemple fictif Thématiques Traçabilité Optimisation d'entrepôt Recommandation de services Partenariats Exemple pédagogique sans partenaires réels.",
    "sha256_text": "6bd67d9f9953a9013bd8ef18836061e4d62c78f486bc356bb635baa81ca1e0a2",
    "character_count": 179,
    "word_count": 19
  }
]''')
print("Exemples de secours :",len(DEMO_DOCUMENTS),"documents FICTIFS")

In [ ]:
MAX_CORPUS_SIZE=20_000_000
MAX_BASELINE_SIZE=20_000_000
MAX_DOCUMENTS=1500
OFFICIAL_HOSTS={'formation.univ-pau.fr','recherche.univ-pau.fr'}


def read_jsonl(raw,limit=MAX_CORPUS_SIZE):
    if len(raw)>limit:raise ValueError('Corpus trop volumineux')
    return [json.loads(line) for line in raw.decode('utf-8-sig').splitlines() if line.strip()]

def archive_entry(path,pattern,limit):
    with zipfile.ZipFile(path) as z:
        matches=[i for i in z.infolist() if re.search(pattern,i.filename) and not i.is_dir()]
        if len(matches)!=1:raise ValueError('Entrée attendue unique introuvable : '+str(pattern))
        entry=matches[0]
        if entry.file_size>limit:raise ValueError('Fichier interne trop volumineux')
        return z.read(entry)

def load_corpus(path):
    p=Path(path)
    if p.suffix.lower()=='.jsonl': return read_jsonl(p.read_bytes())
    if p.suffix.lower()=='.zip':
        return read_jsonl(archive_entry(p,r'(?:^|/)corpus_(?:uppa|synthetic_demo)\.jsonl$',MAX_CORPUS_SIZE))
    raise ValueError('Corpus : ZIP du Notebook 01 ou fichier JSONL attendu')

def load_v2_chunks(path):
    return read_jsonl(archive_entry(Path(path),r'(?:^|/)chunks_(?:uppa|synthetic_demo)\.jsonl$',MAX_BASELINE_SIZE),MAX_BASELINE_SIZE)

DOCS=load_corpus(NB01_PATH) if NB01_PATH else list(DEMO_DOCUMENTS)
KIND_SET={d.get('dataset_kind') for d in DOCS}
assert len(KIND_SET)==1 and KIND_SET.issubset({'live','synthetic_demo'})
KIND=next(iter(KIND_SET))
assert 0<len(DOCS)<=MAX_DOCUMENTS
assert all({'document_id','domain','title','content_markdown','source_url'} <=set(d) for d in DOCS)
assert all(d['domain'] in DOMAINS for d in DOCS)
if KIND=='live':
    for d in DOCS:
        u=urlparse(d.get('source_url') or '')
        assert u.scheme=='https' and u.hostname in OFFICIAL_HOSTS, 'Source réelle non vérifiable'

NAV={'/fr/catalogue/toutes-les-formations.html','/fr/trouver-votre-formation.html',
     '/fr/admission.html','/fr/laboratoires.html'}
INDEX_DOCS=[d for d in DOCS if d.get('document_role')!='navigation_hub'
           and urlparse(d.get('source_url') or '').path not in NAV]
assert INDEX_DOCS
print('Corpus :',KIND,'| documents:',len(DOCS),'| indexables :',len(INDEX_DOCS))

V2_CHUNKS=load_v2_chunks(NB02_V2_PATH) if NB02_V2_PATH else []
if V2_CHUNKS:
    ids={d['document_id'] for d in INDEX_DOCS}
    unknown={c['document_id'] for c in V2_CHUNKS}-ids
    if unknown: raise ValueError(f'La V2 et le corpus NB01 ne correspondent pas : {len(unknown)} IDs inconnus.')
    missing=ids-{c['document_id'] for c in V2_CHUNKS}
    if missing: raise ValueError(f'La V2 et le corpus NB01 ne correspondent pas : {len(missing)} documents absents de V2.')
    if any(c.get('dataset_kind')!=KIND for c in V2_CHUNKS):raise ValueError('Mélange données réelles / fictives')
    BASELINE_KIND='exact_v2_exported'
else:
    BASELINE_KIND='demo_reconstructed' if KIND=='synthetic_demo' else 'missing_v2'
    if KIND=='synthetic_demo':
        # Seulement pour une démonstration : chunks V2 illustratifs, PAS l'export V2 réel.
        for d in INDEX_DOCS:
            src=d['content_markdown'].strip()
            for n,start in enumerate(range(0,len(src),680),1):
                frag=src[start:start+800].strip()
                if not frag:continue
                V2_CHUNKS.append({'chunk_id':f"demo_v2_{d['document_id']}_{n}",
                   'document_id':d['document_id'],'domain':d['domain'],
                   'title':d['title'],'section':'Document','text':frag,
                   'source_url':d.get('source_url'),'dataset_kind':'synthetic_demo',
                   'chunk_type':'prose'})

print('Baseline :',BASELINE_KIND,'| chunks initiaux :',len(V2_CHUNKS))
if KIND=='live' and not V2_CHUNKS:
    raise ValueError('Comparaison réelle impossible sans le ZIP du Notebook 02 V2. Importez les DEUX archives en étape 1.')

### Analyse initiale : où se trouve le déséquilibre ?

Les tailles des pages varient fortement ; une fiche très longue peut produire des centaines de chunks. Cela ne prouve pas qu'elle contient davantage de réponses pertinentes.

In [ ]:
source_df=pd.DataFrame([{'domain':d['domain'],'title':d['title'],'characters':len(d['content_markdown'])}
                       for d in INDEX_DOCS])
display(source_df.groupby('domain').agg(documents=('title','count'),caracteres=('characters','sum'),
          mediane_page=('characters','median')).round(0))
if V2_CHUNKS:
    dist=pd.Series(Counter(c['domain'] for c in V2_CHUNKS),name='chunks_V2').rename_axis('domaine')
    display(dist.to_frame())
    dist.plot(kind='bar',figsize=(7,3),title='Chunks de la V2 par domaine')
    plt.ylabel('Nombre de chunks');plt.tight_layout();plt.show()

---
## Étape 2 — Diagnostic : bruit, doublons et limite de tokens

Une taille de **800 caractères** ne garantit pas une taille acceptable pour le modèle d'embedding. Le texte réellement encodé contient aussi le titre et la section.

Nous utilisons le **même modèle** qu'en V2 afin de rendre l'ablation valable. `encoder.max_seq_length` définit le budget maximum du modèle. Si le texte dépasse cette limite, le risque est de perdre la fin du passage lors de l'encodage.

In [ ]:
encoder=SentenceTransformer(EMBEDDING_MODEL)
MODEL_LIMIT=int(encoder.max_seq_length)
SAFE_TOKEN_LIMIT=min(MODEL_LIMIT,128)
assert SAFE_TOKEN_LIMIT>32
print('Modèle :',EMBEDDING_MODEL,'| dimensions :',encoder.get_sentence_embedding_dimension())
print('Tokens maximum modèle :',MODEL_LIMIT,'| budget pédagogique :',SAFE_TOKEN_LIMIT)

def embedding_text(c):
    return f"Titre : {c['title']}\nSection : {c['section']}\n{c['text']}"

def count_tokens(text):
    return len(encoder.tokenizer.encode(text,add_special_tokens=True,truncation=False))

def noise_reason(text):
    norm=re.sub(r'\s+',' ',text).strip(' #\t\r\n:;–-').casefold()
    if not norm:return 'vide'
    if norm in {'vous êtes ici','vous etes ici','formations','admission','menu','accueil','haut de page',
                'contact','rechercher','recherche','toutes les formations'}:return 'navigation'
    if len(norm)<18 and not any(char.isdigit() for char in norm):return 'fragment très court'
    return ''

if V2_CHUNKS:
    sample=np.unique(np.linspace(0,len(V2_CHUNKS)-1,min(500,len(V2_CHUNKS)),dtype=int))
    diag=pd.DataFrame([{'chars':len(V2_CHUNKS[i]['text']),
        'tokens_total':count_tokens(embedding_text(V2_CHUNKS[i])),
        'bruit':noise_reason(V2_CHUNKS[i]['text']) or 'non'} for i in sample])
    print('Chunks échantillonnés :',len(diag),
          '| risque de troncature :',sum(diag.tokens_total>MODEL_LIMIT),
          '| fragments suspects :',sum(diag.bruit!='non'))
    display(diag.groupby('bruit').size().rename('nombre').to_frame())
    fig,ax=plt.subplots(figsize=(8,3))
    ax.hist(diag.tokens_total,bins=16)
    ax.axvline(MODEL_LIMIT,ls='--',label='Limite du modèle')
    ax.set(xlabel='Tokens (titre + section + texte)',ylabel='Chunks',title='Diagnostic : risque de troncature')
    ax.legend();plt.tight_layout();plt.show()

### À vous de jouer

Repérez au moins **deux fragments trop courts** et un exemple de titre/section qui occupe une part importante du budget de tokens. Discutez : faut-il supprimer un tableau de 40 caractères qui contient un code de formation ? **Non, pas sans vérifier son contexte**.

---
## Étape 3 — Préparer les chunks V3

Trois transformations distinctes :
1. découpage respectueux des titres et des lignes de tableau, sans séparer une ligne courte de son en-tête ;
2. élimination du bruit évident et **fusion prudente des passages courts uniquement à section et document identiques** ;
3. contrôle par le tokenizer : si titre + section + texte sont trop longs, regrouper des **phrases entières** (ou des **lignes de tableau avec leurs en-têtes**) jusqu'au budget de tokens, en conservant les métadonnées.

Les petits tableaux utiles sont gardés ; les éléments de navigation vides ne le sont pas.

In [ ]:
CHAR_SIZE=800
CHAR_OVERLAP=120
MIN_CONTEXT_CHARS=85
TOKEN_OVERLAP=12
splitter=RecursiveCharacterTextSplitter(chunk_size=CHAR_SIZE,chunk_overlap=CHAR_OVERLAP,
             separators=['\n\n','\n','. ',' ', ''],length_function=len,
             keep_separator='end')   # le point reste à la fin de la phrase, pas au début du chunk suivant
HEADING=re.compile(r'^(#{1,6})\s+(.+?)\s*$')
TABLE_LINE=re.compile(r'^\s*\|.*\|\s*$')
TABLE_SEPARATOR=re.compile(r'^\s*\|?\s*:?-{3,}:?\s*(?:\|\s*:?-{3,}:?\s*)+\|?\s*$')


def sections(markdown):
    result=[];hierarchy=[];current=[];label='Introduction'
    def flush():
        if '\n'.join(current).strip(): result.append((label,'\n'.join(current).strip()))
    for line in markdown.splitlines():
        m=HEADING.match(line)
        if m:
            flush();current=[]
            level=len(m.group(1)); hierarchy=[x for x in hierarchy if x[0]<level]
            hierarchy.append((level,m.group(2).strip()));label=' / '.join(v for _,v in hierarchy)
        else:current.append(line)
    flush();return result


def blocks(text):
    out=[];lines=[];kind=None
    def emit():
        if lines:out.append((kind,'\n'.join(lines).strip()))
    for line in text.splitlines():
        t='table' if TABLE_LINE.match(line) else 'prose'
        if not line.strip():emit();lines=[];kind=None;continue
        if kind is not None and kind!=t:emit();lines=[]
        kind=t;lines.append(line)
    emit();return [(k,t) for k,t in out if t]


def split_table(text):
    rows=[x.strip() for x in text.splitlines() if x.strip()]
    if len(rows)<3 or not TABLE_SEPARATOR.match(rows[1]):
        return [('table',x) for x in splitter.split_text(text)]
    prefix=rows[0]+'\n'+rows[1]
    data=rows[2:];buffer=[];output=[]
    def emit():
        if buffer:output.append(('table',prefix+'\n'+'\n'.join(buffer)))
    for ix,row in enumerate(data,1):
        candidate=prefix+'\n'+'\n'.join(buffer+[row])
        if len(candidate)<=CHAR_SIZE:
            buffer.append(row);continue
        emit();buffer=[]
        if len(prefix+'\n'+row)<=CHAR_SIZE:
            buffer.append(row);continue
        # ligne surdimensionnée : garder l'index de ligne et les en-têtes
        headers=[x.strip() for x in rows[0].strip('|').split('|')]
        values=[x.strip() for x in row.strip('|').split('|')]
        line=' ; '.join(f'{headers[i] if i<len(headers) else "Colonne"}: {v}' for i,v in enumerate(values))
        for part in splitter.split_text(line):
            output.append(('table_row_fragment',f'Ligne {ix}: '+part))
    emit()
    return output if output else [('table',prefix)]


def structural_chunks(doc):
    output=[]
    for label,body in sections(doc['content_markdown']):
        for kind,text in blocks(body):
            if kind=='table':segments=split_table(text)
            else:segments=[('prose',x) for x in splitter.split_text(text)]
            for subtype,segment in segments:
                if not noise_reason(segment) or subtype!='prose':
                    output.append({'section':label,'chunk_type':subtype,'text':segment})
    return output

# Tests structurels (fictifs) : en-têtes préservés, ligne longue fragmentée
TEST_TABLE='| Code | Enseignement |\n| --- | --- |\n'+ '\n'.join(
    f'| UE{i} | Données industrielles et systèmes intelligents '+('long texte '*90)+' |' for i in range(3))
parts=split_table(TEST_TABLE)
assert len(parts)>2 and any('Ligne' in text for kind,text in parts)
print('Tests de tableau :',len(parts),'fragments — OK')

In [ ]:
# Fusion très prudente : uniquement des fragments PROSE voisins du même document et de la même section.
def fuse_short(pieces):
    output=[];fusions=0
    for p in pieces:
        p=dict(p)
        if (output and p['chunk_type']=='prose' and output[-1]['chunk_type']=='prose'
            and p['section']==output[-1]['section']
            and (len(p['text'])<MIN_CONTEXT_CHARS or len(output[-1]['text'])<MIN_CONTEXT_CHARS)
            and len(p['text'])+len(output[-1]['text'])+2<=CHAR_SIZE):
            output[-1]['text']+='\n\n'+p['text'];fusions+=1
        else:output.append(p)
    return output,fusions

PROVISIONAL=[];fused_total=0
for d in INDEX_DOCS:
    pcs,n=fuse_short(structural_chunks(d));fused_total+=n
    for p in pcs:
        PROVISIONAL.append({'document_id':d['document_id'],'domain':d['domain'],
            'title':d['title'],'source_url':d.get('source_url'),
            'collected_at':d.get('collected_at'),'academic_years_detected':d.get('academic_years_detected',[]),
            'dataset_kind':KIND,'document_role':d.get('document_role','content_page'),
            **p})
assert PROVISIONAL
print('Chunks structurels provisoires :',len(PROVISIONAL),'| fusions prudentes :',fused_total)
print('Répartition :',dict(Counter(c['domain'] for c in PROVISIONAL)))

### Découper en tokens sans perdre la provenance

`SentenceTransformer` a une longueur maximum propre au modèle. Pour chaque passage, on mesure les tokens de **`Titre + Section + contenu`**. Le tokenizer sert uniquement à **compter** : le texte conservé reste celui de la source.

- **Prose :** on regroupe des phrases entières ; la dernière phrase d'un fragment est reprise au début du suivant lorsqu'elle est courte (chevauchement).
- **Tableaux :** on regroupe des lignes entières et **chaque fragment répète la ligne d'en-têtes** ; une ligne trop longue est réécrite en `colonne : valeur`.
- **Dernier recours :** une phrase plus longue que le budget est coupée entre deux mots.

> Pourquoi ne pas découper la liste de tokens puis la re-décoder ? `tokenizer.decode` ne restitue pas les retours à la ligne : les tableaux seraient aplatis, et l'extrait cité ne correspondrait plus mot pour mot à la page source.

In [ ]:
def content_tokens(text):
    """Nombre de tokens du texte seul (sans tokens spéciaux). Le tokenizer ne sert qu'à compter."""
    return len(encoder.tokenizer.encode(text,add_special_tokens=False,truncation=False))

SENTENCE_END=re.compile(r'(?<=[.!?…;])\s+')

def prose_units(text):
    """Phrases de la source, dans l'ordre ; un saut de ligne final marque la fin d'une ligne."""
    units=[]
    for line in text.split('\n'):
        if not line.strip():continue
        parts=[x for x in SENTENCE_END.split(line.strip()) if x]
        parts[-1]+='\n'
        units.extend(parts)
    return units

def render_prose(units):
    return ''.join(u if u.endswith('\n') else u+' ' for u in units).strip()

def split_words(unit,budget):
    """Dernier recours : une phrase plus longue que le budget est coupée ENTRE deux mots."""
    pieces=[];current=[]
    for word in unit.split():
        while content_tokens(word)>budget and len(word)>1:   # « mot » démesuré (URL, code)
            cut=len(word)//2
            if current:pieces.append(' '.join(current));current=[]
            head,word=word[:cut],word[cut:]
            pieces.extend(split_words(head,budget))
        if current and content_tokens(' '.join(current+[word]))>budget:
            pieces.append(' '.join(current));current=[]
        current.append(word)
    if current:pieces.append(' '.join(current))
    return pieces

def pack_units(units,budget,render,overlap_tokens=0):
    """Regroupe des unités ENTIÈRES tant que le rendu tient dans le budget de tokens."""
    groups=[];current=[]
    for unit in units:
        if current and content_tokens(render(current+[unit]))>budget:
            groups.append(current)
            last=current[-1]
            keep=(overlap_tokens and len(current)>1 and content_tokens(last)<=overlap_tokens
                  and content_tokens(render([last,unit]))<=budget)
            current=[last] if keep else []
        current.append(unit)
    if current:groups.append(current)
    return [render(g) for g in groups]

def split_prose_by_tokens(text,budget):
    units=[]
    for unit in prose_units(text):
        if content_tokens(unit)<=budget:units.append(unit)
        else:
            parts=split_words(unit,budget)
            if unit.endswith('\n') and parts:parts[-1]+='\n'
            units.extend(parts)
    return pack_units(units,budget,render_prose,overlap_tokens=TOKEN_OVERLAP)

def split_table_by_tokens(text,budget):
    """Retourne [(type, fragment)] : chaque fragment de tableau répète ses en-têtes."""
    rows=[x.strip() for x in text.splitlines() if x.strip()]
    if len(rows)<3 or not TABLE_SEPARATOR.match(rows[1]):
        return [('table',x) for x in split_prose_by_tokens(text,budget)]
    header=rows[0]+'\n'+rows[1]
    render=lambda group:header+'\n'+'\n'.join(group)
    output=[];pending=[]
    def flush():
        if pending:
            output.extend(('table',x) for x in pack_units(pending,budget,render))
            pending.clear()
    names=[x.strip() for x in rows[0].strip('|').split('|')]
    for ix,row in enumerate(rows[2:],1):
        if content_tokens(render([row]))<=budget:
            pending.append(row);continue
        flush()   # ligne trop longue : la réécrire « colonne : valeur », coupée entre deux mots
        values=[x.strip() for x in row.strip('|').split('|')]
        line=' ; '.join(f'{names[n] if n<len(names) else "Colonne"} : {v}' for n,v in enumerate(values))
        label=f'Ligne {ix} — '
        room=max(8,budget-content_tokens(label))
        output.extend(('table_row_fragment',label+x) for x in split_words(line,room))
    flush()
    return output

def token_safe_chunks(p,token_budget=SAFE_TOKEN_LIMIT):
    prefix=f"Titre : {p['title']}\nSection : {p['section']}\n"
    payload_budget=token_budget-count_tokens(prefix)-4
    source_item=dict(p)
    if payload_budget<20:
        # Métadonnées complètes conservées dans des champs supplémentaires ;
        # les champs title/section utilisés par le modèle doivent rester compatibles
        # avec la reconstruction du prompt du Notebook 03.
        source_item['full_title']=p['title']
        source_item['full_section']=p['section']
        source_item['title']=p['title'][:85]
        source_item['section']=p['section'].split(' / ')[-1][-55:]
        prefix=f"Titre : {source_item['title']}\nSection : {source_item['section']}\n"
        payload_budget=token_budget-count_tokens(prefix)-4
        if payload_budget<12:
            raise ValueError('Titre de document trop long pour le budget de tokens ; revoir les métadonnées.')
    if p['chunk_type']=='table':
        fragments=split_table_by_tokens(p['text'],payload_budget)
    elif p['chunk_type']=='table_row_fragment' and re.match(r'Ligne \d+\s*[:—]\s*',p['text']):
        # Ligne de tableau géante : chaque fragment garde son étiquette « Ligne n ».
        label=re.match(r'Ligne \d+\s*[:—]\s*',p['text']).group(0)
        room=max(8,payload_budget-content_tokens(label))
        fragments=[('table_row_fragment',label+x) for x in split_prose_by_tokens(p['text'][len(label):],room)]
    else:
        fragments=[(p['chunk_type'],x) for x in split_prose_by_tokens(p['text'],payload_budget)]
    result=[]
    for n,(kind,frag) in enumerate(fragments):
        frag=frag.strip()
        # Garde-fou : la concaténation préfixe + texte peut coûter un ou deux tokens de plus.
        while count_tokens(prefix+frag)>token_budget and ' ' in frag:
            frag=frag.rsplit(' ',1)[0]
        if not frag:continue
        item=dict(source_item);item['text']=frag;item['chunk_type']=kind
        item['embedding_prefix']=prefix
        item['token_count']=count_tokens(prefix+frag)
        item['char_count']=len(frag)
        item['token_fragment_start']=n   # rang du fragment dans le passage d'origine
        result.append(item)
    return result

# Tests (fictifs) : fidélité à la source, en-têtes répétés, budget respecté
_demo={'title':'Fiche de test','section':'Programme','chunk_type':'table',
       'text':'| Code | Enseignement |\n| --- | --- |\n'+'\n'.join(
           f'| UE{i:02d} | Données industrielles, capteurs et logistique numéro {i} |' for i in range(30))}
_parts=token_safe_chunks(_demo)
assert len(_parts)>1 and all(x['text'].startswith('| Code | Enseignement |\n| --- | --- |') for x in _parts)
assert all(x['token_count']<=SAFE_TOKEN_LIMIT for x in _parts)
assert sum(x['text'].count('| UE') for x in _parts)==30, 'aucune ligne perdue ni dupliquée'
_prose={'title':'Fiche de test','section':'Objectifs','chunk_type':'prose',
        'text':' '.join(f'La phrase numéro {i} décrit une compétence du parcours.' for i in range(40))}
_pp=token_safe_chunks(_prose)
assert len(_pp)>1 and all(x['text'].endswith('.') for x in _pp), 'coupures en fin de phrase uniquement'
assert all(x['text'] in _prose['text'] for x in _pp), 'chaque fragment est un extrait exact de la source'
print('Tests du découpage par tokens : OK —',len(_parts),'fragments de tableau,',len(_pp),'fragments de prose')

V3_CHUNKS=[];order=Counter()
for p in PROVISIONAL:
    for frag in token_safe_chunks(p):
        # Le rang dans le document entre dans l'identifiant : un texte répété dans la même
        # section (ligne de tableau, phrase type) ne doit pas produire deux fois le même ID.
        order[frag['document_id']]+=1
        frag['chunk_order']=order[frag['document_id']]
        frag['chunk_id']='v3_'+sha256((frag['document_id']+'|'+frag['section']+'|'+
                              frag['text']+'|'+str(frag['chunk_order'])).encode()).hexdigest()[:24]
        frag['oversize_table']=False
        # Champs facultatifs présents en V2 et attendus par les ateliers suivants
        frag['table_index']=None;frag['table_row_start']=None
        frag['table_row_fragment']=frag['chunk_type']=='table_row_fragment'
        V3_CHUNKS.append(frag)
assert V3_CHUNKS and len(set(c['chunk_id'] for c in V3_CHUNKS))==len(V3_CHUNKS)
assert all(c['token_count']<=SAFE_TOKEN_LIMIT for c in V3_CHUNKS)
print('V2 :',len(V2_CHUNKS),'| V3 :',len(V3_CHUNKS),
      '| tokens maximum V3 :',max(c['token_count'] for c in V3_CHUNKS))

---
## Étape 4 — Contrôler la qualité des nouveaux chunks

Une amélioration ne signifie pas forcément une réduction du nombre de chunks. Le découpage en tokens peut au contraire augmenter leur nombre. Ce qui compte : **moins de troncature, moins de bruit et une meilleure récupération**.

In [ ]:
def chunk_audit(chunks,label):
    small=sum(len(c['text'])<45 for c in chunks)
    noisy=sum(bool(noise_reason(c['text'])) for c in chunks)
    bad_token=sum(count_tokens(embedding_text(c) if 'embedding_prefix' not in c else
                     c['embedding_prefix']+c['text'])>SAFE_TOKEN_LIMIT for c in chunks)
    return {'version':label,'chunks':len(chunks),'fragments_<45c':small,'suspects_bruit':noisy,
            'risque_troncature':bad_token,'documents':len({c['document_id'] for c in chunks})}

audit=pd.DataFrame([chunk_audit(V2_CHUNKS,'V2 référence'),chunk_audit(V3_CHUNKS,'V3 optimisée')])
display(audit)
by_domain=pd.concat([pd.DataFrame({'version':'V2','domain':[x['domain'] for x in V2_CHUNKS]}),
                     pd.DataFrame({'version':'V3','domain':[x['domain'] for x in V3_CHUNKS]})])
ct=by_domain.groupby(['domain','version']).size().unstack(fill_value=0)
display(ct)
ct.plot(kind='bar',figsize=(8,3.5))
plt.title('Comparaison du volume documentaire par domaine');plt.ylabel('Chunks')
plt.tight_layout();plt.show()

### Exercice d'analyse

Si le nombre de chunks V3 augmente, faut-il abandonner V3 ? Interprétez les valeurs de `risque_troncature` et la répartition par domaine avant de conclure. Relevez un tableau pertinent dont il ne faut pas supprimer les lignes courtes.

---
## Étape 5 — Construire deux index indépendants dans Qdrant

La V2 exécutée auparavant **n'est jamais modifiée**. Cette séance recrée son index depuis ses chunks exportés dans un dossier spécifique au Notebook 02 V3. Les deux collections utilisent les **mêmes embeddings**, la même distance cosine et le même corpus source.

Les modèles locaux Qdrant offrent `QdrantClient(path=...)`, `create_collection`, `upsert` et `query_points`. On n'utilise aucun serveur distant. Cf. [documentation Qdrant](https://qdrant.tech/documentation/quickstart/).

In [ ]:
# La V2 est remise dans un index indépendant, jamais dans son dossier originel.
# La V3 utilise son propre index et son propre nom de collection.
if 'client' in globals():
    try:client.close()
    except Exception:pass
client=QdrantClient(path=str(DB))
NAME_V2=f'uppa_{KIND}_v2_reference'
NAME_V3=f'uppa_{KIND}_chunks_v3'
DIM=int(encoder.get_sentence_embedding_dimension())


def vectors_for(chunks):
    texts=[c.get('embedding_prefix',f"Titre : {c['title']}\nSection : {c['section']}\n")+c['text']
           for c in chunks]
    a=encoder.encode(texts,normalize_embeddings=True,convert_to_numpy=True,batch_size=24,show_progress_bar=True)
    a=np.asarray(a,dtype=np.float32)
    assert a.shape==(len(chunks),DIM) and np.all(np.isfinite(a))
    return a


def create_index(name,chunks,vectors):
    if client.collection_exists(name):client.delete_collection(name)
    client.create_collection(collection_name=name,
             vectors_config=models.VectorParams(size=DIM,distance=models.Distance.COSINE))
    for start in range(0,len(chunks),64):
        pts=[]
        for c,vec in zip(chunks[start:start+64],vectors[start:start+64]):
            point_id=str(uuid.uuid5(uuid.NAMESPACE_URL,c['chunk_id']))
            payload={k:v for k,v in c.items() if k not in ('embedding_prefix',)}
            payload['embedding_model']=EMBEDDING_MODEL
            pts.append(models.PointStruct(id=point_id,vector=vec.tolist(),payload=payload))
        client.upsert(collection_name=name,points=pts,wait=True)
    assert client.get_collection(name).points_count==len(chunks)
    print(name,'→',len(chunks),'points')

start=time.perf_counter()
VECTORS_V2=vectors_for(V2_CHUNKS)
VECTORS_V3=vectors_for(V3_CHUNKS)
create_index(NAME_V2,V2_CHUNKS,VECTORS_V2)
create_index(NAME_V3,V3_CHUNKS,VECTORS_V3)
print('Construction totale (secondes) :',round(time.perf_counter()-start,1))

---
## Étape 6 — Recherche globale et diversification des documents

Une recherche top-k renvoie des **chunks**, pas nécessairement k sources différentes. Nous construisons donc une récupération avec un maximum de deux passages par document, puis une diversification équilibrée par domaine. Ces quotas **ne garantissent pas la pertinence**.

In [ ]:
def raw_search(query,index,domain=None,limit=24):
    qvec=encoder.encode(query,normalize_embeddings=True)
    filt=(models.Filter(must=[models.FieldCondition(key='domain',
          match=models.MatchValue(value=domain))]) if domain else None)
    response=client.query_points(collection_name=index,query=np.asarray(qvec,dtype=np.float32).tolist(),
                         query_filter=filt,limit=limit,with_payload=True)
    return [{'score':float(hit.score),**dict(hit.payload or {})} for hit in response.points]


def unique_documents(hits,k=6,max_per_doc=1):
    output=[];perdoc=Counter()
    for h in hits:
        if perdoc[h['document_id']]>=max_per_doc:continue
        output.append(h);perdoc[h['document_id']]+=1
        if len(output)>=k:break
    return output


def search_global(question,index,k=6):
    return unique_documents(raw_search(question,index,limit=max(24,k*10)),k=k)


def search_balanced(question,index,k=6,domains=DOMAINS):
    per={d:unique_documents(raw_search(question,index,domain=d,limit=max(16,k*4)),k=k)
         for d in domains}
    out=[];seen=set();p=0
    while len(out)<k:
        added=False
        for d in domains:
            if p<len(per[d]):
                h=per[d][p]
                if h['document_id'] not in seen:
                    out.append(h);seen.add(h['document_id']);added=True
                    if len(out)>=k:break
        if not added:break
        p+=1
    return out

Q_CROSS=('Quelles formations abordent les systèmes industriels, comment candidater '
         'en master et quels laboratoires travaillent sur les systèmes intelligents ?')
for label,index in [('V2 — globale',NAME_V2),('V3 — globale',NAME_V3)]:
    hits=search_global(Q_CROSS,index)
    print(label,dict(Counter(h['domain'] for h in hits)),
          [(h['title'][:55],round(h['score'],3)) for h in hits[:3]])

---
## Étape 7 — Décomposer une question transversale sans LLM

L'étudiant peut d'abord tester une décomposition **explicite et compréhensible**. La fonction ci-dessous se base sur des mots-clés, et non sur une prétendue compréhension sémantique par un agent : ses erreurs doivent être observées. Lorsqu'aucun thème n'est reconnu, la sous-question reprend la question d'origine : aucun sujet n'est ajouté à la place de l'utilisateur.

Exemple de question : *« Quelles formations portent sur les systèmes industriels, comment candidater en master et quels laboratoires correspondent à cette spécialité ? »*

In [ ]:
def decompose_question(question):
    t=question.casefold()
    # Règles transparentes et modifiables : à évaluer expérimentalement.
    formation=any(w in t for w in ('formation','parcours','diplôme','master','étude','programme'))
    admission=any(w in t for w in ('admission','candidat','inscri','accès','dossier','calendrier'))
    recherche=any(w in t for w in ('laboratoire','recherche','doctorat','scientifiq','thématique'))
    themes=[]
    for token in ('systèmes industriels','systèmes intelligents','logistique','informatique',
                  'maintenance','intelligence artificielle','optimisation','capteurs'):
        if token in t:themes.append(token)
    # Sans thème reconnu, on garde la question d'origine : ne JAMAIS injecter un sujet
    # que l'utilisateur n'a pas mentionné (cela orienterait les résultats).
    sujet=' et '.join(themes[:3])
    queries={}
    if formation:queries['formation']=(f'Quelles formations et quels parcours abordent {sujet} ?'
                                       if sujet else question)
    if admission:
        niveau='master 2' if any(w in t for w in ('master 2','m2','deuxième année')) else 'master'
        queries['admission']=f'Quelles conditions, pièces et procédures de candidature pour le {niveau} ?'
    if recherche:queries['recherche']=(f'Quels laboratoires et thèmes de recherche concernent {sujet} ?'
                                       if sujet else question)
    if not queries:
        # Ne pas imposer trois domaines lorsqu'un seul semble nécessaire.
        queries={'formation':question}
    return queries


# Ajustements MANUELS, réglés en observant quelques requêtes : ce ne sont pas des paramètres
# appris. Ils doivent être validés sur des questions annotées qui n'ont PAS servi à les régler.
BONUS_MASTER=0.08
MALUS_HORS_MASTER=0.20

def admission_rescore(hit,subquestion):
    text=(hit.get('title','')+' '+hit.get('section','')+' '+hit.get('text','')[:350]).casefold()
    new_score=hit['score']
    if 'master' in subquestion.casefold():
        if 'master' in text:new_score+=BONUS_MASTER
        if re.search(r'(école d.ingénieur|\bl2\b|\bl3\b|licence professionnelle)',text):new_score-=MALUS_HORS_MASTER
    return new_score


def search_decomposed(question,index,k=6,rescore=True):
    subs=decompose_question(question)
    order=list(subs)
    candidates={}
    for domain,subq in subs.items():
        items=unique_documents(raw_search(subq,index,domain=domain,limit=max(20,k*5)),k=k)
        items=[{**h,'ranking_score':admission_rescore(h,subq) if (rescore and domain=='admission') else h['score'],
                'subquery':subq} for h in items]
        candidates[domain]=sorted(items,key=lambda h:h['ranking_score'],reverse=True)
    selected=[];seen=set()
    for rank in range(k):
        progressed=False
        for domain in order:
            for h in candidates[domain]:
                if h['document_id'] not in seen:
                    selected.append(h);seen.add(h['document_id']);progressed=True
                    break
            if len(selected)>=k:break
        if len(selected)>=k or not progressed:break
    return selected[:k],subs

subqueries=decompose_question(Q_CROSS)
print('Sous-questions explicites :')
for domain,q in subqueries.items():print('-',domain,':',q)
method_D,_=search_decomposed(Q_CROSS,NAME_V3,k=6)
display(pd.DataFrame([{'domain':h['domain'],'title':h['title'],'score':round(h['score'],3),
                       'source_url':h.get('source_url')} for h in method_D]))

### Expérience A / B / C / D

Les méthodes suivantes utilisent **exactement la même question**. Toutes fournissent `k` documents distincts, pour comparer la diversité sans confondre 6 passages d'une fiche et 6 fiches.

In [ ]:
K=6
STRATEGIES={
    'A_V2_globale': lambda q:search_global(q,NAME_V2,K),
    'B_V3_globale': lambda q:search_global(q,NAME_V3,K),
    'C_V3_diversifiee': lambda q:search_balanced(q,NAME_V3,K),
    'D_V3_decomposee': lambda q:search_decomposed(q,NAME_V3,K)[0],
    'D0_V3_decomposee_sans_bonus': lambda q:search_decomposed(q,NAME_V3,K,rescore=False)[0],
}
measure=[]
for name,fn in STRATEGIES.items():
    t0=time.perf_counter();found=fn(Q_CROSS);latency=time.perf_counter()-t0
    measure.append({'strategie':name,'documents':len(found),'domaines':', '.join(sorted({x['domain'] for x in found})),
                    'latence_s':round(latency,3),'titres_top3':' | '.join(h['title'][:34] for h in found[:3])})
display(pd.DataFrame(measure))
print('Une meilleure diversité n’est PAS automatiquement une meilleure pertinence.')

---
## Étape 8 — Questions annotées : évaluation document-level

**Protocole obligatoire :** lire les pages sources et annoter les `document_id` pertinents **avant** de regarder les scores des stratégies. Ne renseignez pas les références « au hasard » en prenant les résultats du moteur : cela fausserait l'évaluation.

Importez facultativement `questions_a_annoter_uppa.csv` complété (issu de V2), ou utilisez le modèle proposé ci-dessous. Tant que les références sont vides, **aucun score global n'est affiché sur les données réelles**.

In [ ]:
ANNOTATIONS_CSV_PATH = ''  # chemin vers le CSV annoté, après vérification humaine
QUESTION_BANK=[
 ('formation','Quels parcours sont proposés dans le Master Informatique ?'),
 ('formation','Quels programmes abordent les capteurs ou les systèmes industriels ?'),
 ('formation','Quelles compétences décrit le parcours Industry 4.0 ?'),
 ('admission','Comment candidater à un master 1 ?'),
 ('admission','Comment candidater à un master 2 ?'),
 ('admission','Quelle période de candidature est indiquée pour une année précisée ?'),
 ('recherche','Quels laboratoires travaillent sur les systèmes intelligents ?'),
 ('recherche','Quelles thématiques scientifiques portent sur l’informatique ?'),
 ('transversal','Quelles formations portent sur les systèmes industriels, comment candidater et quels laboratoires existent ?'),
 ('transversal','Quelles formations en logistique correspondent à des thématiques de recherche en optimisation ?'),
]

if KIND=='live':
    template=pd.DataFrame([{'question_id':f'Q{i:02d}','domain_cible':d,'question':q,
         'reference_document_ids':'','justification':'','statut_annotation':'a_annoter'}
         for i,(d,q) in enumerate(QUESTION_BANK,1)])
    template.to_csv(EXPORT/'questions_a_annoter_uppa_v3.csv',index=False,encoding='utf-8-sig')
    catalog=pd.DataFrame([{'document_id':d['document_id'],'domain':d['domain'],
          'title':d['title'],'source_url':d.get('source_url')} for d in INDEX_DOCS])
    catalog.to_csv(EXPORT/'catalogue_documents_uppa_v3.csv',index=False,encoding='utf-8-sig')
    print('Grille à compléter :',EXPORT/'questions_a_annoter_uppa_v3.csv')
    print('Catalogue des IDs :',EXPORT/'catalogue_documents_uppa_v3.csv')
    display(template.head())
else:
    # ÉTIQUETTES réservées aux six documents fictifs — aucune revendication sur l'UPPA.
    def demo_id(title):
        return [d['document_id'] for d in INDEX_DOCS if title.casefold() in d['title'].casefold()]
    label_rows=[('Quels cours parlent des capteurs ?',demo_id('Industry 4.0')),
                ('Quels documents sont demandés pour le master ?',demo_id('Admission en master')),
                ('Quel document présente un calendrier ?',demo_id('Calendrier')),
                ('Quels laboratoires travaillent sur les réseaux ?',demo_id('Laboratoire Systèmes')),
                ('Quel projet aborde la traçabilité ?',demo_id('Logistique intelligente'))]
    template=pd.DataFrame([{'question_id':f'D{i:02d}','domain_cible':'tous','question':q,
            'reference_document_ids':';'.join(ids),'justification':'Exemple fictif',
            'statut_annotation':'demo_fictive'} for i,(q,ids) in enumerate(label_rows,1) if ids])
    print('⚠️ Références de démonstration FICTIVES ; aucun score ne décrit le site UPPA.')
    display(template)

In [ ]:
if ANNOTATIONS_CSV_PATH:
    labels=pd.read_csv(ANNOTATIONS_CSV_PATH,dtype=str).fillna('')
else:
    labels=template.copy()

valid_ids={d['document_id'] for d in INDEX_DOCS}

def parse_refs(value):
    if isinstance(value,list):return set(value)
    if not isinstance(value,str):return set()
    return {s.strip() for s in value.split(';') if s.strip()}

EVAL_CASES=[]
for row in labels.to_dict('records'):
    refs=parse_refs(row.get('reference_document_ids',''))
    if not refs:continue
    unknown=refs-valid_ids
    if unknown:raise ValueError(f'Identifiants de référence inconnus : {sorted(unknown)}')
    EVAL_CASES.append({'question':row['question'],'refs':refs,'question_id':row.get('question_id','')})
print('Questions réellement annotées :',len(EVAL_CASES),'/',len(labels))
if KIND=='live' and len(EVAL_CASES)<3:
    print('⚠️ Scores institutionnels NON CALCULÉS : annoter au moins 3 questions manuellement.')

In [ ]:
def metrics_at_k(found,refs,k=K):
    # DOCUMENTS distincts : tous les retrievers ont le même budget de documents.
    ids=[h['document_id'] for h in found[:k]]
    hits=[rank for rank,d in enumerate(ids,1) if d in refs]
    return {'Hit@k':int(bool(hits)),
            'Recall@k':len(refs.intersection(ids))/len(refs),
            'MRR@k':1/min(hits) if hits else 0.0}

rows=[]
if EVAL_CASES and (KIND!='live' or len(EVAL_CASES)>=3):
    for case in EVAL_CASES:
        for label,fn in STRATEGIES.items():
            t0=time.perf_counter();results=fn(case['question']);duration=time.perf_counter()-t0
            rows.append({'question_id':case['question_id'],'question':case['question'],
                         'strategie':label,'latence_s':duration,
                         **metrics_at_k(results,case['refs'],K)})
EVAL_DF=pd.DataFrame(rows)
if EVAL_DF.empty:
    print('Pas de résultats expérimentaux : le jeu de vérité terrain est incomplet.')
else:
    summary=EVAL_DF.groupby('strategie').agg({'Hit@k':'mean','Recall@k':'mean',
         'MRR@k':'mean','latence_s':'mean'}).round(3)
    display(summary)
    summary[['Hit@k','Recall@k','MRR@k']].plot(kind='bar',figsize=(9,3.5),ylim=(0,1))
    plt.title('Métriques au niveau document (annotations manuelles ou démo fictive)')
    plt.ylabel('Moyenne');plt.tight_layout();plt.show()
    EVAL_DF.to_csv(EXPORT/'evaluation_comparative_v2_v3.csv',index=False,encoding='utf-8-sig')

### Discussion scientifique

- **A → B :** même requête, même modèle, index différent : quel effet du chunking ?
- **B → C :** même index V3 : quel effet de la diversification ?
- **C → D :** que gagne ou perd la décomposition heuristique ?
- **D0 → D :** les bonus d'admission réglés à la main (`BONUS_MASTER`, `MALUS_HORS_MASTER`) aident-ils sur des questions qui n'ont pas servi à les régler ?
- Une stratégie peut augmenter **Recall@k** mais dégrader **MRR@k** ou la latence.
- Il faut analyser les erreurs de la recherche Admissions (M1/M2, écoles d'ingénieurs, licences) et les dates périmées.
- Une variation mesurée sur 3–5 questions fictives ne constitue **pas une preuve générale de supériorité**.

---
## Étape 9 — Exporter l'index V3 pour le Notebook 03

La structure de l'archive est la même que celle de V2 (`exports/index_manifest.json`, `exports/chunks_*.jsonl`, `qdrant_local/`). Le champ `schema_version=uppa_rag_index_v1` est **conservé par compatibilité** avec le Notebook 03, tandis que `experiment_version='NB02_V3'` identifie la nouvelle stratégie.

⚠️ La V2 de référence est conservée dans sa propre archive d'origine ; le ZIP V3 exporte **seulement l'index optimisé**, pas l'index de comparaison.

In [ ]:
SUFFIX='uppa' if KIND=='live' else 'synthetic_demo'
CHUNK_PATH=EXPORT/f'chunks_{SUFFIX}.jsonl'
with CHUNK_PATH.open('w',encoding='utf-8') as f:
    for c in V3_CHUNKS:
        item={k:v for k,v in c.items() if k!='embedding_prefix'}
        f.write(json.dumps(item,ensure_ascii=False)+'\n')

AUDIT_PATH=EXPORT/'audit_chunking_v2_v3.csv'
audit.to_csv(AUDIT_PATH,index=False,encoding='utf-8-sig')
metadata={
  'schema_version':'uppa_rag_index_v1',
  'experiment_version':'NB02_V3',
  'created_at':datetime.now(timezone.utc).isoformat(timespec='seconds'),
  'dataset_kind':KIND,'source_schema':'uppa_rag_doc_v1',
  'collection_name':NAME_V3,'embedding_model':EMBEDDING_MODEL,
  'embedding_dimension':DIM,'distance':'COSINE',
  'normalized_embeddings':True,'chunking':'structure_aware_token_safe_v3',
  'chunk_size_characters':CHAR_SIZE,'chunk_overlap_characters':CHAR_OVERLAP,
  'chunk_max_tokens_with_context':SAFE_TOKEN_LIMIT,
  'n_documents':len(INDEX_DOCS),'n_chunks':len(V3_CHUNKS),
  'domains':dict(Counter(c['domain'] for c in V3_CHUNKS)),
  'baseline_provenance':BASELINE_KIND,
  'note': 'Corpus FICTIF, exercices uniquement' if KIND!='live' else
          'Corpus public UPPA ; vérifier l’actualité des pages et les conditions d’admission',
}
(EXPORT/'index_manifest.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('Chunks prêts pour Notebook 03 :',CHUNK_PATH)

In [ ]:
# Pour exporter uniquement l'index V3, reconstituer une base locale avec une seule collection.
# La base de comparaison contient V2+V3 ; elle n'est PAS exportée afin d'éviter de fausses ambiguïtés.
client.close()
EXPORT_DB=BASE/'export_nb03'/'qdrant_local'
EXPORT_DB.parent.mkdir(parents=True,exist_ok=True)
if EXPORT_DB.exists():shutil.rmtree(EXPORT_DB)
nb03_client=QdrantClient(path=str(EXPORT_DB))
nb03_client.create_collection(collection_name=NAME_V3,
       vectors_config=models.VectorParams(size=DIM,distance=models.Distance.COSINE))
for start in range(0,len(V3_CHUNKS),64):
    batch=[]
    for c,vec in zip(V3_CHUNKS[start:start+64],VECTORS_V3[start:start+64]):
        item={k:v for k,v in c.items() if k!='embedding_prefix'}
        item['embedding_model']=EMBEDDING_MODEL
        batch.append(models.PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL,c['chunk_id'])),
                                        vector=vec.tolist(),payload=item))
    nb03_client.upsert(collection_name=NAME_V3,points=batch,wait=True)
assert nb03_client.get_collection(NAME_V3).points_count==len(V3_CHUNKS)
nb03_client.close()

archive=BASE/f'UPPA_Notebook02_V3_index_{SUFFIX}.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(EXPORT.rglob('*')):
        if path.is_file(): z.write(path,Path('exports')/path.relative_to(EXPORT))
    for path in sorted(EXPORT_DB.rglob('*')):
        if path.is_file(): z.write(path,Path('qdrant_local')/path.relative_to(EXPORT_DB))
print('Index V3 :',archive,'|',round(archive.stat().st_size/1e6,2),'Mo')
print('Pour télécharger :')
print(f'from google.colab import files; files.download({str(archive)!r})')

---
## Étape 10 — Travail à rendre et passage au Notebook 03

**Rapport court à rendre :**

1. Une description des documents réels exploités et des limites du corpus.
2. Un tableau comparant V2/V3 : chunks, risques de troncature, fragments trop courts, domaines.
3. Trois requêtes métiers pour chaque master (Industry 4.0 et Logistique), dont au moins une question transversale.
4. Un jeu de **références annotées manuellement** et une analyse des métriques : Hit@6, Recall@6, MRR@6.
5. Deux cas où la V3 améliore le retrieval, et un cas où elle échoue ou régresse (si ces cas existent).
6. Une justification des paramètres de chunking, de `k`, du choix des filtres et du budget de tokens.

**Pour Notebook 03 :** chargez l'archive `UPPA_Notebook02_V3_index_uppa.zip` dans sa cellule d'import. Le schéma est compatible, mais le Notebook 03 détecte désormais aussi les archives `UPPA_Notebook02_V3_index_*.zip` ; si une archive V2 et une archive V3 sont présentes, indiquez explicitement celle à utiliser dans `INPUT_ZIP`.

### Ressources
- [Qdrant — client Python, mode local et requêtes filtrées](https://qdrant.tech/documentation/quickstart/)
- [Sentence Transformers — modèle, tokenizer et embeddings](https://sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html)

*Ce notebook est un prototype pédagogique, indépendant de l'UPPA. Aucun résultat de démonstration fictive ne doit être présenté comme une donnée officielle.*